# `03-geocoding-umpd-incidents.ipynb` : Using Geocodio.io to geocode as many locations we can without having to rely on Google

## Setup

In [1]:
import pandas as pd 
import geopandas as gpd
import folium as fl
import requests
import os
import json
import time
import re
from pathlib import Path
from umpd_crime.cleaning import is_standard_address, clean_location, classify_change
from umpd_crime.geocoding import google_geocode_addrs, get_shapely_point, create_markers_for_flmap
from shapely import Point

DATA_DIR = Path.cwd().parent / "data"
CURRENT_YEAR = time.localtime().tm_year

GEOCODIO_BASE_URL = "https://api.geocod.io/v2.1/geocode/"
GOOGLE_BASE_URL = "https://maps.googleapis.com/maps/api/geocode/json"

HEADERS = {
  "User-Agent": "Jour720/1.0 (diegojto@umd.edu)",
  "Accept-Language": "en-US,en;q=0.9",
}

geocodio_params = {
  "q": None,
  "api_key" : os.getenv("UMPDCRIMEGEOCODIOAPIKEY")
}

google_params = {
  'address': None,
  'bounds': '38.883790,-77.065390|39.081929,-76.810495',
  'key': os.getenv("UMPDCRIMEGOOGLEGEOCODINGAPIKEY")
}

df = pd.read_csv(DATA_DIR / "processed" / f"merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv")

In [2]:
if not Path(DATA_DIR / "processed" / "geocoding").is_dir():
  os.mkdir(Path(DATA_DIR / "processed" / "geocoding"))
  geocoding_dir = Path(DATA_DIR / "processed" / "geocoding")
else:
  i = 1
  while Path(DATA_DIR / "processed" / "geocoding" / f"geocoding_{i}").is_dir():
    i += 1
  os.mkdir(Path(DATA_DIR / "processed" / "geocoding"/ f"geocoding_{i}"))
  geocoding_dir = Path(DATA_DIR / "processed" / "geocoding" / f"geocoding_{i}" )

In [2]:
cases_and_locs_df = df[['umpd_case_number', 'location']].drop_duplicates()
#cases_and_locs_df.to_csv(geocoding_dir / "ci_locations_raw.csv")

In [3]:
df['location_og_standard'] = df['location'].apply(is_standard_address)
df['location_clean'] = df['location'].apply(clean_location)
df['location_clean_standard'] = df['location_clean'].apply(is_standard_address)
df['loc_cleaning_result'] = df.apply(lambda row: classify_change(
  row['location'],
  row['location_clean']
), axis=1)
# df[['location', 'location_clean']].drop_duplicates().to_csv(DATA_DIR / "processed" / "geocoding" / geocoding_dir / "locations_compare.csv")

In [4]:
cleaning_result_categories = df['loc_cleaning_result'].unique().tolist()
cleaning_result_counts = df['loc_cleaning_result'].value_counts().rename_axis("cleaning_result").reset_index(name='num_rows').set_index('cleaning_result')

total_locs_changed = len(df) - int(cleaning_result_counts.at['Unchanged', 'num_rows'])
perc_locs_changed = total_locs_changed / len(df) * 100 

print(cleaning_result_counts, "\n")
print(f"Total % locs changed: {perc_locs_changed:.2f}%")
print(f"# of nunique locations % change: {((df['location_clean'].nunique() - df['location'].nunique()) / df['location'].nunique()) * 100:.2f}%")

                                               num_rows
cleaning_result                                        
Unchanged                                          8681
Address extracted from a building description      4558
Intersection standardized                            26
Coordinates extracted                                 4
Same-road pair reduced to one road                    2 

Total % locs changed: 34.59%
# of nunique locations % change: -13.83%


In [5]:
cleaned_locations = df['location_clean'].unique().tolist()

In [6]:
response = requests.get(GEOCODIO_BASE_URL, params={
  "q": "Campus Dr and Rossborough Ln, College Park, Maryland",
  "api_key": os.getenv("UMPDCRIMEGEOCODIOAPIKEY")
}, headers = HEADERS)

response.json()

{'results': [{'stable_address_key': 'gcod_uss8agpe3lnyyup3',
   'address_components': {'street': 'Campus',
    'suffix': 'Dr',
    'formatted_street': 'Campus Dr',
    'city': 'College Park',
    'county': "Prince George's County",
    'state_province': 'MD',
    'postal_code': '20742',
    'country': 'US'},
   'address_lines': ['Campus Dr', '', 'College Park, MD 20742'],
   'formatted_address': 'Campus Dr, College Park, MD 20742',
   'location': {'lat': 38.986747, 'lng': -76.94665},
   'accuracy': 1,
   'accuracy_type': 'street_center',
   'match_type': None,
   'source': 'TIGER/Line® from the US Census Bureau'}]}

In [7]:
df.head()

,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,crime_type,...,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges,location_og_standard,location_clean,location_clean_standard,loc_cleaning_result
0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,Fireworks Complaint,...,NaN,Asian/Pacific Islander,Male,Fireworks Discharge without the required permit,1.0,['Fireworks Discharge without the required per...,True,4300 block of Knox Rd,True,Unchanged
1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,DWI/DUI,...,NaN,Black,Female,"(Driving, Attempting to drive) veh. while impa...",6.0,"['(Driving, Attempting to drive) veh. while im...",True,8300 block of Baltimore Ave,True,Unchanged
2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,DWI/DUI,...,NaN,Asian/Pacific Islander,Male,"(Driving, Attempting to drive) veh. while impa...",2.0,"['(Driving, Attempting to drive) veh. while im...",True,7400 block of Baltimore Ave,True,Unchanged
3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,Vandalism,...,NaN,NaN,NaN,NaN,NaN,NaN,True,7500 block of Calvert Service Ln,True,Unchanged
4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,CDS Violation,...,NaN,Black,Male,CDS:Possession Marijuana L/T 10 grams,1.0,['CDS:Possession Marijuana L/T 10 grams'],False,Metzerott Rd,False,Unchanged


In [8]:
df['arrest_date'] = pd.to_datetime(df['arrest_date'])
df_2025 = df[df['arrest_date'].dt.year == 2025]
df_2026 = df[df['arrest_date'].dt.year == 2026]
cases_and_locs_2025 = df_2025[['umpd_case_number', 'location']]
cases_and_locs_2026 = df_2026[["umpd_case_number", 'location']]

In [9]:
cases_and_locs_2026['location+CPMD'] = cases_and_locs_2026['location'] + " College Park, MD"
cases_and_locs_2026.to_csv(Path(DATA_DIR / "processed" / "geocoding" / "cases_and_locs_2026.csv"))

## Google Geocoding API

The Google Geocoding API supports adding a `bounds` parameter that biases the geocoder's results to be within a defined area. 

I used bboxfinder to create an around 13 km box with College Park in the center, and it gave me these bounds: `38.883790,-77.065390,39.081929,-76.810495`.

Here's what a request for one address looks like: 

In [11]:
google_params['address'] = "Baltimore Ave and Iribe Center"

response = requests.get(GOOGLE_BASE_URL, params=google_params, headers=HEADERS)

data = response.json()
data

{'results': [{'address_components': [{'long_name': '8125',
     'short_name': '8125',
     'types': ['street_number']},
    {'long_name': 'Paint Branch Drive',
     'short_name': 'Paint Branch Dr',
     'types': ['route']},
    {'long_name': 'College Park',
     'short_name': 'College Park',
     'types': ['locality', 'political']},
    {'long_name': "Prince George's County",
     'short_name': "Prince George's County",
     'types': ['administrative_area_level_2', 'political']},
    {'long_name': 'Maryland',
     'short_name': 'MD',
     'types': ['administrative_area_level_1', 'political']},
    {'long_name': 'United States',
     'short_name': 'US',
     'types': ['country', 'political']},
    {'long_name': '20742', 'short_name': '20742', 'types': ['postal_code']}],
   'formatted_address': '8125 Paint Branch Dr, College Park, MD 20742, USA',
   'geometry': {'location': {'lat': 38.9891232, 'lng': -76.9364707},
    'location_type': 'ROOFTOP',
    'viewport': {'northeast': {'lat': 38.9

Let's do this now for all unique location strings.

In [12]:
google_geocode_results = google_geocode_addrs(cleaned_locations, HEADERS)

Data did not return with correct JSON structure for Lot 8; setting None for latitude and longitude
Data did not return with correct JSON structure for Lot 9b; setting None for latitude and longitude
Data did not return with correct JSON structure for Lot 3; setting None for latitude and longitude
Data did not return with correct JSON structure for Lot SS2; setting None for latitude and longitude
Data did not return with correct JSON structure for Recreation Artificial Turf Field, Turf Field, Turf Fields; setting None for latitude and longitude
Data did not return with correct JSON structure for Lot GG1; setting None for latitude and longitude
Data did not return with correct JSON structure for Lot A; setting None for latitude and longitude
25% geocoding done
Data did not return with correct JSON structure for Marathon; setting None for latitude and longitude
Data did not return with correct JSON structure for Route 95/route 201; setting None for latitude and longitude
Data did not retu

In [14]:
google_geocode_results_gdf = gpd.GeoDataFrame(google_geocode_results, geometry = 'point', crs="EPSG:4326")
google_geocode_results_gdf.to_csv(DATA_DIR / "processed" / "geocoding" / f"google_geocoding_{CURRENT_YEAR}{time.localtime().tm_mon}{time.localtime().tm_mday}_results.csv",index=False)

NameError: name 'google_geocode_results' is not defined

In [10]:
google_geocode_results_gdf = gpd.read_file(DATA_DIR / 'processed' / 'geocoding' / f"google_geocoding_{CURRENT_YEAR}{time.localtime().tm_mon}{time.localtime().tm_mday}_results.csv") 
google_geocode_results_gdf['latitude'] = pd.to_numeric(google_geocode_results_gdf['latitude'])
google_geocode_results_gdf['longitude'] = pd.to_numeric(google_geocode_results_gdf['longitude'])

In [11]:
google_geocode_results_gdf = google_geocode_results_gdf.set_geometry(gpd.points_from_xy(google_geocode_results_gdf['longitude'], google_geocode_results_gdf['latitude']), crs="EPSG:4326").drop('point', axis=1) 

In [15]:
google_geocode_results_gdf2 = google_geocode_results_gdf.drop('field_1', axis=1)
google_geocode_results_gdf2

,source_location,latitude,longitude,geometry
0,4300 block of Knox Rd,38.981487,-76.942387,POINT (-76.94239 38.98149)
1,8300 block of Baltimore Ave,38.992458,-76.933607,POINT (-76.93361 38.99246)
2,7400 block of Baltimore Ave,38.980983,-76.938134,POINT (-76.93813 38.98098)
3,7500 block of Calvert Service Ln,38.982283,-76.942474,POINT (-76.94247 38.98228)
4,Metzerott Rd,39.002939,-76.956240,POINT (-76.95624 39.00294)
...,...,...,...,...
918,4200 block of Metzerott Rd,38.999649,-76.935903,POINT (-76.9359 38.99965)
919,40 W Baltimore National Pkwy,39.290502,-76.610407,POINT (-76.61041 39.2905)
920,E 32 Eb Just Before 95 Exit Hwy,NaN,NaN,POINT (NaN NaN)
921,8700 block of 35th Ave,38.998956,-76.948923,POINT (-76.94892 38.99896)


In [16]:
umd_center = [38.987218, -76.942511]
umd_center_gs = gpd.GeoSeries([Point(umd_center[1], umd_center[0])], crs="EPSG:4326")
umd_map = fl.Map(location = umd_center, zoom_start = 13.5)
create_markers_for_flmap(google_geocode_results_gdf2, umd_map, "blue")

In [ ]:
umd_map.save(Path(Path.cwd().parent / 'outputs' / 'maps' / "google_geocoding_first_attempt.html").absolute())

In [74]:
utm_crs = google_geocode_results_gdf2.estimate_utm_crs()
google_geocode_results_projected = google_geocode_results_gdf2.to_crs(utm_crs)
umd_center_projected = umd_center_gs.to_crs(utm_crs).iloc[0]

distances = google_geocode_results_projected.distance(umd_center_projected)
radius_meters = 300000 # 30km
outside_count = (distances > radius_meters).sum()
print(f"Number of locations Google geocoded to be more than 30km away from UMD: {int(outside_count)}")

Number of locations Google geocoded to be more than 30km away from UMD: 54


In [23]:
df = pd.merge(df, google_geocode_results_gdf2, how='left', left_on = 'location_clean', right_on='source_location')
df.to_csv(DATA_DIR / 'processed' / 'geocoding' / 'merged_df_w_google_geocodes.csv', index=False)

## Geocodio

In [ ]:
results = []
start_time = time.time()
for i, case_num, location in tuple(cases_and_locs_2025.itertuples()):
  
  result_dict = {"umpd_case_number": case_num, "source_location": location + " College Park, MD"}
  
  if i == 0: 
    print("Beginning geocoding")
  elif i == len(cases_and_locs_df) // 2:
    print("Halfway done")
  geocodio_params['q'] = location + " College Park, MD"
  response = requests.get(GEOCODIO_BASE_URL, params=geocodio_params, headers=HEADERS)
  if response.status_code != 200: 
    print(f"Error geocoding {location}, setting to None")
    print(f"Received {response.status_code}, saying {response.reason}")
    result_dict['geocodio_formatted_address'] = None
    result_dict['geocodio_lat'] = None
    result_dict['geocodio_long'] = None
    result_dict['geocodio_accuracy'] = None
    result_dict['geocodio_accuracy_type'] = None
    result_dict['geocodio_match_type'] = None
  else: 
    data = response.json()
    try:
      data = data['results'][0]
    except IndexError as e:
      print("Didn't return expected structure:")
      print(data['results'])
      print("Setting options to None")
      result_dict['geocodio_formatted_address'] = None
      result_dict['geocodio_lat'] = None
      result_dict['geocodio_long'] = None
      result_dict['geocodio_accuracy'] = None
      result_dict['geocodio_accuracy_type'] = None
      result_dict['geocodio_match_type'] = None
    else: 
      result_dict['geocodio_formatted_address'] = data['formatted_address'] if data['formatted_address'] else "",
      result_dict["geocodio_lat"] = data['location']['lat'] if data['location']['lat'] else 0.0,
      result_dict["geocodio_long"] = data['location']['lng'] if data['location']['lng'] else 0.0,
      result_dict['geocodio_accuracy'] = data['accuracy'] if data['accuracy'] else 0,
      result_dict['geocodio_accuracy_type'] = data['accuracy_type'] if data['accuracy_type'] else "",
      result_dict['geocodio_match_type'] = data['match_type'] if data['match_type'] else ""
    
  results.append(result_dict)
print("Done")
end_time = time.time()
execution_time = end_time - start_time
print(f"Geocoding {len(cases_and_locs_2025)} addresses took {execution_time} seconds")

Done
Geocoding 12912 addresses took 92.83824825286865 seconds


In [31]:
geocodio_2025_results_df = pd.DataFrame(results)
geocodio_2025_results_df

,umpd_case_number,source_location,geocodio_formatted_address,geocodio_lat,geocodio_long,geocodio_accuracy,geocodio_accuracy_type,geocodio_match_type
0,2021-00081808,"64, Dorchester, Dorchester Hall at 3947 Campus...","(64 West Dr, College Park, MD 20740,)","(39.029713,)","(-76.941253,)","(0.87,)","(range_interpolation,)",
1,2023-00038780,"Baltimore Ave College Park, MD","(Baltimore Ave, College Park, MD 20740,)","(39.00021,)","(-76.93121,)","(1,)","(street_center,)",
2,2023-00055167,"Laboratory for Physical Sciences, LPS at 8050 ...","(8050 Greenmead Dr, College Park, MD 20740,)","(39.00349,)","(-76.943676,)","(1,)","(rooftop,)",building_centroid
3,2023-00079735,"A James Clark Hall, A. James Clark Hall, A.J. ...","(Paint Branch Dr, College Park, MD 20740,)","(38.998712,)","(-76.941425,)","(1,)","(street_center,)",
4,2023-00086320,"Commons 4, SCC 4, South Campus Commons 4 at 43...","(4 4th St, Laurel, MD 20707,)","(39.107653,)","(-76.846698,)","(0.7,)","(range_interpolation,)",
...,...,...,...,...,...,...,...,...
422,2025-00283429,"University Blvd College Park, MD","(University Blvd, College Park, MD 20740,)","(39.00009,)","(-76.943231,)","(1,)","(street_center,)",
423,2025-00283504,"Baltimore Ave College Park, MD","(Baltimore Ave, College Park, MD 20740,)","(39.00021,)","(-76.93121,)","(1,)","(street_center,)",
424,2025-00286752,"Baltimore Ave College Park, MD","(Baltimore Ave, College Park, MD 20740,)","(39.00021,)","(-76.93121,)","(1,)","(street_center,)",
425,2025-00290080,"University Blvd College Park, MD","(University Blvd, College Park, MD 20740,)","(39.00009,)","(-76.943231,)","(1,)","(street_center,)",


In [32]:
df.head()

,Unnamed: 0,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,...,arrest_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges
0,0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,...,21888.0,2019-01-01 00:01:00,Not during a semester,Not during a semester,NaN,Asian/Pacific Islander,Male,Fireworks Discharge without the required permit,1.0,['Fireworks Discharge without the required per...
1,1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,...,21895.0,2019-01-01 01:37:00,Not during a semester,Not during a semester,NaN,Black,Female,"(Driving, Attempting to drive) veh. while impa...",6.0,"['(Driving, Attempting to drive) veh. while im..."
2,2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,...,21896.0,2019-01-01 01:41:00,Not during a semester,Not during a semester,NaN,Asian/Pacific Islander,Male,"(Driving, Attempting to drive) veh. while impa...",2.0,"['(Driving, Attempting to drive) veh. while im..."
3,3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,...,21889.0,2019-01-02 23:30:00,Winter 2019,Winter,NaN,Black,Male,CDS:Possession Marijuana L/T 10 grams,1.0,['CDS:Possession Marijuana L/T 10 grams']
